In [0]:
%python
data=((1,'Naval','Solapur'),(2,'Aditya','Mumbai'))
schema="id int, name string, city string"

df=spark.createDataFrame(data,schema)
df.write.mode("overwrite").saveAsTable("dev.bronze.cyntexa_emp")

In [0]:
select * from dev.silver.cyntexa_emp_silver

In [0]:
%python
data=((1,'Naval','Delhi'),(3,'Rahul','Jaipur'))
schema="id int, name string, city string"

df=spark.createDataFrame(data,schema)
df.write.mode("overwrite").saveAsTable("dev.bronze.cyntexa_emp")

In [0]:
select * from dev.bronze.cyntexa_emp

In [0]:
merge into dev.silver.cyntexa_emp_silver t
using dev.bronze.cyntexa_emp s
on t.id = s.id
when matched then update set t.name = s.name, t.city = s.city
when not matched then insert *

In [0]:
select * from dev.silver.cyntexa_emp_silver

In [0]:
%python
data=((1,'Naval','Solapur'),(2,'Aditya','Mumbai'))
schema="id int, name string, city string"

df=spark.createDataFrame(data,schema)
df.write.mode("overwrite").saveAsTable("dev.bronze.cyntexa_emp")

In [0]:
%python
data=((1,'Naval','Delhi'),(3,'Rahul','Jaipur'))
schema="id int, name string, city string"

df=spark.createDataFrame(data,schema)
df.write.mode("overwrite").saveAsTable("dev.bronze.cyntexa_emp")

## SCD Type 2 Implementation

This implements Slowly Changing Dimension Type 2 to track historical changes:

**Steps to test:**
1. Run Cell 9 to create the target table
2. Run Cell 7 to load initial data (Naval-Solapur, Aditya-Mumbai)
3. Run Cells 10-11 to perform initial load
4. Run Cell 8 to load changed data (Naval-Delhi, Rahul-Jaipur)
5. Run Cells 10-11 again to capture changes
6. Run Cell 12 to see the history

**Expected result:**
- Initial load: 2 records (Naval-Solapur, Aditya-Mumbai) with is_current=true
- After change: 4 records total
  - Naval-Solapur: is_current=false, end_date set (closed)
  - Naval-Delhi: is_current=true (new version)
  - Aditya-Mumbai: is_current=true (unchanged)
  - Rahul-Jaipur: is_current=true (new record)

In [0]:
-- Create SCD Type 2 target table with history tracking columns
CREATE TABLE IF NOT EXISTS dev.silver.cyntexa_emp_scd2 (
  id INT,
  name STRING,
  city STRING,
  effective_date DATE,
  end_date DATE,
  is_current BOOLEAN
)

In [0]:
-- SCD Type 2 MERGE: Track historical changes
MERGE INTO dev.silver.cyntexa_emp_scd2 AS t
USING (
  SELECT id, name, city
  FROM dev.bronze.cyntexa_emp
  QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY id) = 1
) AS s
ON t.id = s.id AND t.is_current = true

-- When matched and data changed: close current record
WHEN MATCHED AND (t.name != s.name OR t.city != s.city) THEN
  UPDATE SET 
    t.is_current = false,
    t.end_date = current_date()

-- When not matched: insert new record as current
WHEN NOT MATCHED THEN
  INSERT (id, name, city, effective_date, end_date, is_current)
  VALUES (s.id, s.name, s.city, current_date(), null, true)

In [0]:
-- SCD Type 2 Part 2: Insert new versions of changed records
INSERT INTO dev.silver.cyntexa_emp_scd2
SELECT 
  s.id,
  s.name,
  s.city,
  current_date() AS effective_date,
  NULL AS end_date,
  true AS is_current
FROM (
  SELECT id, name, city
  FROM dev.bronze.cyntexa_emp
  QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY id) = 1
) s
INNER JOIN dev.silver.cyntexa_emp_scd2 t
  ON s.id = t.id 
  AND t.end_date = current_date()
  AND (t.name != s.name OR t.city != s.city)

In [0]:
-- View SCD Type 2 history: shows all versions of records
SELECT * 
FROM dev.silver.cyntexa_emp_scd2
ORDER BY id, effective_date